# ASR test: Whisper large-v3-turbo with transformers
- Model: https://huggingface.co/openai/whisper-large-v3-turbo
- Goal: check that the model runs locally on CPU and measure its speed on FLEURS.
- Runs the original model (safetensors) with PyTorch, without CTranslate2 conversion.
- Environment: `uv sync`, then select the `.venv` (Python 3.12) kernel.

## 1. Setup

In [1]:
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

c:\Users\alexi\HES-Hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Device and precision
- No NVIDIA GPU (no CUDA) on this laptop, so the model runs on CPU in FP32.

In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

## 2. Model

#### Model loading
- The first run downloads the model (~1.6 GB) to the Hugging Face cache; later runs load it from disk.
- Run this cell once; the model stays in memory until the kernel restarts.

In [3]:
model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 587/587 [00:03<00:00, 191.06it/s]


#### Pipeline
- Wraps the whole flow: audio -> log-mel spectrogram -> model -> tokens -> text.
- `return_timestamps=True` returns segment-level timestamps in `result["chunks"]`; use `"word"` for word-level.
- `help(pipe)` lists all parameters of the installed version.

In [4]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True
)

#help(pipe)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


## 3. Smoke test (LibriSpeech)
- `distil-whisper/librispeech_long`: about 1 minute of clean English audiobook speech.
- Only checks that transcription works: the dataset has no reference text, so WER cannot be computed on it.
- Decoding the audio needs `torchcodec`, which in turn needs the FFmpeg shared libraries (e.g. `winget install --id Gyan.FFmpeg.Shared`). Restart the kernel after installing packages.

In [5]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
type(dataset)


datasets.arrow_dataset.Dataset

In [6]:
sample = dataset[0]["audio"]

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, 8, and 9, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.14.1+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 9:
Traceback (most recent ca

- The language is auto-detected here; it can be forced with `generate_kwargs={"language": "fr", "task": "transcribe"}`.

In [ ]:
result = pipe(sample)
result["text"]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

" Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel. Nor is Mr. Quilter's manner less interesting than his matter. He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similes drawn from eating and its results occur most readily to the mind. He has grave doubts whether Sir Frederick Layton's work is really Greek after all, and can discover in it but little of rocky Ithaca. Linnell's pictures are a sort of Up Guards and Adam paintings, and Mason's exquisite idles are as national as a jingo poem. Mr. Birkett Foster's landscapes smile at one much in the same way that Mr. Carker used to flash his teeth, and Mr. John Collier gives his sitter a cheerful slap on the back before he says like a shampooer in a Turkish bath next man"

## 4. Speed on FLEURS
- FLEURS (Few-shot Learning Evaluation of Universal Representations of Speech): https://huggingface.co/datasets/google/fleurs
- Same Wikipedia sentences read in many languages; here the first 20 validation clips in English and French.
- Only the validation parquet file is downloaded (the full config includes a ~2 GB train split).

**Metrics**
- WER (Word Error Rate): measures how accurate the model is. The model's output is compared word-for-word with the correct text (reference).
- RTF (Real-Time Factor): processing time / audio duration. Below 1 means faster than real time.

In [ ]:
N_SAMPLES = 20

def load_fleurs(lang_config, n=N_SAMPLES):
    # load only the validation parquet file, not the whole config (train is ~2 GB)
    url = f"hf://datasets/google/fleurs/parquet-data/{lang_config}/validation-00000-of-00001.parquet"
    ds = load_dataset("parquet", data_files=url, split="train")  # "train" is just the default name here
    return ds.select(range(n))

fleurs = {
    "en": load_fleurs("en_us"),
    "fr": load_fleurs("fr_fr"),
}

sample = fleurs["fr"][0]
sample["raw_transcription"], sample["audio"]["sampling_rate"]

Generating train split: 394 examples [00:00, 551.50 examples/s]
Generating train split: 289 examples [00:00, 656.81 examples/s]


("Il y a de nombreuses répercussions sociales et politiques telles que l'utilisation du système métrique, le passage de l'absolutisme au républicanisme, le nationalisme et la conviction que le pays appartient au peuple et non à un seul dirigeant.",
 16000)

- One warm-up call first (not measured), then every clip is timed with the language forced.

In [ ]:
def transcribe(lang, ds):
    rows = []
    for i, ex in enumerate(ds):
        duration = ex["num_samples"] / 16000          # FLEURS audio is 16 kHz
        t0 = time.perf_counter()
        out = pipe(ex["audio"], generate_kwargs={"language": lang, "task": "transcribe"})
        elapsed = time.perf_counter() - t0
        rows.append({
            "lang": lang,
            "id": ex["id"],
            "reference": ex["raw_transcription"],
            "hypothesis": out["text"].strip(),
            "duration_s": duration,
            "time_s": elapsed,
        })
        print(f"{lang} {i + 1}/{len(ds)}  audio {duration:5.1f}s  took {elapsed:5.1f}s")
    return rows

# warm-up: the first call is slower, keep it out of the measurement
_ = pipe(fleurs["en"][0]["audio"], generate_kwargs={"language": "en", "task": "transcribe"})

results = transcribe("en", fleurs["en"]) + transcribe("fr", fleurs["fr"])

en 1/20  audio   6.5s  took  27.6s
en 2/20  audio  16.4s  took  41.1s
en 3/20  audio   7.9s  took  37.9s
en 4/20  audio   4.1s  took  27.7s
en 5/20  audio   4.9s  took  26.4s
en 6/20  audio   4.4s  took  27.1s
en 7/20  audio   6.8s  took  29.3s
en 8/20  audio  10.0s  took  34.7s
en 9/20  audio   9.3s  took  28.4s
en 10/20  audio   9.5s  took  29.9s
en 11/20  audio   7.3s  took  27.7s
en 12/20  audio   8.2s  took  29.7s
en 13/20  audio   7.1s  took  28.5s
en 14/20  audio   7.5s  took  27.9s
en 15/20  audio  10.1s  took  29.3s
en 16/20  audio  17.5s  took  30.5s
en 17/20  audio  12.6s  took  28.3s
en 18/20  audio   8.5s  took  26.2s
en 19/20  audio   7.8s  took  25.9s
en 20/20  audio   6.8s  took  25.4s
fr 1/20  audio  14.5s  took  35.2s
fr 2/20  audio   6.0s  took  31.8s
fr 3/20  audio  15.4s  took  43.3s
fr 4/20  audio   7.9s  took  29.8s
fr 5/20  audio   8.5s  took  24.9s
fr 6/20  audio   7.4s  took  30.1s
fr 7/20  audio  12.0s  took  33.5s
fr 8/20  audio   6.9s  took  27.8s
fr 9/20  

In [ ]:
df = pd.DataFrame(results)

summary = df.groupby("lang")[["duration_s", "time_s"]].sum()
summary["rtf"] = summary["time_s"] / summary["duration_s"]
summary

,duration_s,time_s,rtf
lang,,,
en,173.38,589.300292,3.398894
fr,188.70,595.618082,3.156429


**Result (2026-10-20, CPU, FP32):** RTF 3.40 in English and 3.16 in French, about 3x slower than real time.
The faster-whisper comparison and WER are in `model-comparison.ipynb`.

- Save the transcriptions in the same format as `model-comparison.ipynb`, so this run can be added to its WER table.
- Needs `results` from the cell above in memory (re-running costs ~20 minutes).

In [ ]:
df.assign(model="large-v3-turbo-transformers", compute_type="float32", load_s=float("nan")) \
  .to_csv("results/fleurs_large-v3-turbo-transformers_float32.csv", index=False)